In [ ]:
# ============================================================
# 🦾 MASTER DASHBOARD: All-In-One (2D Report + 3D Replay)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D 
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import os
import re

# 1. Setup
BASE_LOG_DIR = os.path.join("..", "..", "logs")
L1 = 15.0 # cm
L2 = 15.0 # cm
MAX_ALLOWED_ERROR = 0.5 # cm

def load_and_analyze(run_folder_path):
    clear_output(wait=True)
    run_name = os.path.basename(run_folder_path)
    
    # --- LOAD DATA ---
    log_files = glob.glob(os.path.join(run_folder_path, "arm_2link_log.csv"))
    if not log_files:
        print(f"❌ No data found in {run_name}")
        return
    df = pd.read_csv(log_files[0])
    df['time_sec'] = df['timestamp_ms'] / 1000.0
    
    # --- PRE-CALCULATE GEOMETRY ---
    rad_m1 = np.radians(df['motor1_pos'])
    df['elbow_x'] = L1 * np.cos(rad_m1)
    df['elbow_y'] = L1 * np.sin(rad_m1)
    
    # --- METRICS ---
    duration = df['time_sec'].max()
    max_error = df['error'].max()
    avg_error = df['error'].mean()
    verdict = "PASS" if max_error <= MAX_ALLOWED_ERROR else "FAIL"
    
    total_j1_travel = np.sum(np.abs(np.diff(df['motor1_pos'])))
    total_j2_travel = np.sum(np.abs(np.diff(df['motor2_pos'])))
    
    dt = df['time_sec'].diff().fillna(0.001)
    # Skip first 5 frames for velocity calculation to avoid startup noise
    max_j1_vel = (np.abs(df['motor1_pos'].diff()) / dt).iloc[5:].max()
    max_j2_vel = (np.abs(df['motor2_pos'].diff()) / dt).iloc[5:].max()

    # ========================================================
    # 📊 PART 1: STATIC ANALYSIS REPORT (The 3 Plots)
    # ========================================================
    fig_static = plt.figure(figsize=(12, 12))
    plt.subplots_adjust(hspace=0.4)
    
    # Plot 1: Cartesian Path
    ax1 = fig_static.add_subplot(3, 1, 1)
    ax1.set_title(f"1. Cartesian Path ({run_name})", fontweight='bold')
    ax1.plot(df['target_x'], df['target_y'], 'g--', label='Target')
    ax1.plot(df['real_x'], df['real_y'], 'b-', label='Actual Tip')
    # Draw Final Pose
    last = df.iloc[-1]
    ax1.plot([0, last['elbow_x'], last['real_x']], 
             [0, last['elbow_y'], last['real_y']], 'r-o', lw=2, label='Final Pose')
    ax1.scatter([0], [0], c='k', s=100, marker='s') # Base
    ax1.axis('equal')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')
    ax1.set_ylabel("Y (cm)")

    # Plot 2: Error Timeline
    ax2 = fig_static.add_subplot(3, 1, 2)
    ax2.set_title("2. Tracking Error (Full Run)", fontweight='bold')
    ax2.plot(df['time_sec'], df['error'], 'r-', label='Error')
    ax2.axhline(MAX_ALLOWED_ERROR, color='k', linestyle=':', label='Limit')
    ax2.fill_between(df['time_sec'], df['error'], color='red', alpha=0.1)
    ax2.set_ylabel("Error (cm)")
    ax2.grid(True, alpha=0.3)

    # Plot 3: Landing Zoom (Final 20% of time)
    ax3 = fig_static.add_subplot(3, 1, 3)
    start_zoom = duration * 0.8
    zoom_df = df[df['time_sec'] > start_zoom]
    ax3.set_title("3. Landing Zoom (Final 20%)", fontweight='bold')
    if not zoom_df.empty:
        ax3.plot(zoom_df['time_sec'], zoom_df['error'], 'r-o', markersize=4)
        ax3.axhline(0, color='k', alpha=0.5)
        # Auto-scale Y axis for detail
        y_limit = max(zoom_df['error'].max(), 0.05) * 1.2
        ax3.set_ylim(-0.01, y_limit)
    ax3.set_ylabel("Error (cm)")
    ax3.set_xlabel("Time (s)")
    ax3.grid(True, which='both', alpha=0.3)

    plt.show()
    plt.close(fig_static) # Clean up memory

    # --- SUMMARY TABLE ---
    summary_data = {
        "Metric": [
            "Duration", "Max Error", "Avg Error", "Verdict", 
            "---",
            "J1 Total Travel", "J2 Total Travel",
            "J1 Max Speed", "J2 Max Speed"
        ],
        "Value": [
            f"{duration:.2f} s", f"{max_error:.4f} cm", f"{avg_error:.4f} cm", verdict,
            "---",
            f"{total_j1_travel:.1f}°", f"{total_j2_travel:.1f}°",
            f"{max_j1_vel:.1f} deg/s", f"{max_j2_vel:.1f} deg/s"
        ]
    }
    print("\n" + "="*60)
    print(f"📊 REPORT: {run_name}")
    print("="*60)
    print(pd.DataFrame(summary_data).to_string(index=False))
    print("="*60 + "\n")

    # ========================================================
    # 🎮 PART 2: 3D INTERACTIVE REPLAY
    # ========================================================
    print("👇 3D INTERACTIVE REPLAY (Use Sliders to Rotate & Move) 👇")
    
    def plot_3d_frame(frame_index, elev, azim):
        row = df.iloc[frame_index]
        history_slice = df.iloc[:frame_index+1:5] # Sample trace for speed
        
        # Create a specific figure for the 3D plot
        fig_3d = plt.figure(figsize=(10, 8))
        ax = fig_3d.add_subplot(111, projection='3d')
        
        # 1. Setup 3D Stage
        ax.set_xlim(-10, 35)
        ax.set_ylim(-15, 25)
        ax.set_zlim(0, 20)
        ax.set_xlabel('X (cm)')
        ax.set_ylabel('Y (cm)')
        ax.set_zlabel('Z (Height)')
        ax.set_title(f"T={row['time_sec']:.2f}s | J1: {row['motor1_pos']:.0f}° J2: {row['motor2_pos']:.0f}°")
        
        # 2. Draw "Floor" Grid
        ax.plot([-10, 35, 35, -10, -10], [-15, -15, 25, 25, -15], [0,0,0,0,0], 'k-', alpha=0.1)
        
        # 3. Paths
        ax.plot(df['target_x'], df['target_y'], np.zeros(len(df)), 'g--', alpha=0.3, label="Target")
        ax.plot(history_slice['real_x'], history_slice['real_y'], np.zeros(len(history_slice)), 
                'b-', linewidth=1, alpha=0.6, label="Trace")

        # 4. Robot Arm (3D Structure)
        arm_z = 5.0 # Lift it up
        
        # Base Tower
        ax.plot([0,0], [0,0], [0, arm_z], 'k-', linewidth=6, alpha=0.5) 
        
        # Link 1
        ax.plot([0, row['elbow_x']], [0, row['elbow_y']], [arm_z, arm_z], 
                color='gray', linewidth=4, solid_capstyle='round')
        
        # Link 2
        ax.plot([row['elbow_x'], row['real_x']], [row['elbow_y'], row['real_y']], [arm_z, arm_z], 
                color='red', linewidth=4, solid_capstyle='round', label="Arm")
        
        # Shadow
        ax.plot([0, row['elbow_x'], row['real_x']], 
                [0, row['elbow_y'], row['real_y']], 
                [0, 0, 0], 'k-', linewidth=2, alpha=0.2)

        # Joints
        ax.scatter([0], [0], [arm_z], c='black', s=100)
        ax.scatter([row['elbow_x']], [row['elbow_y']], [arm_z], c='black', s=80)
        
        ax.view_init(elev=elev, azim=azim)
        ax.legend(loc='upper left')
        plt.show()
        plt.close(fig_3d)

    # Define Sliders
    slider_time = widgets.IntSlider(min=0, max=len(df)-1, step=10, value=0, description='Time:', layout=widgets.Layout(width='60%'))
    slider_elev = widgets.IntSlider(min=0, max=90, step=5, value=30, description='Elevation:')
    slider_azim = widgets.IntSlider(min=0, max=360, step=5, value=270, description='Rotation:')
    
    # Layout Controls
    ui = widgets.VBox([
        slider_time, 
        widgets.HBox([slider_elev, slider_azim])
    ])
    
    # Create Output Link
    out = widgets.interactive_output(plot_3d_frame, {
        'frame_index': slider_time, 
        'elev': slider_elev, 
        'azim': slider_azim
    })
    
    display(ui, out)

# --- FOLDER DROPDOWN ---
all_folders = [f.path for f in os.scandir(BASE_LOG_DIR) if f.is_dir()]
arm_folders = sorted(
    [f for f in all_folders if "arm_run" in os.path.basename(f)],
    key=lambda x: int(re.search(r"arm_run_?(\d+)", os.path.basename(x)).group(1)) 
                  if re.search(r"arm_run_?(\d+)", os.path.basename(x)) else 0
)

if not arm_folders:
    print("❌ No 'arm_run' folders found.")
else:
    w = widgets.Dropdown(options=[(os.path.basename(f), f) for f in arm_folders], description='Select Run:')
    widgets.interact(load_and_analyze, run_folder_path=w)

interactive(children=(Dropdown(description='Select Run:', options=(('arm_run_1', '..\\..\\logs\\arm_run_1'), (…